# Flow - Matching

This file contains logic for:
- design of a network architecture
- t embedding
- transferring a ResNet as a encoder
- receives t(r) and class data, and a sample from the designed Linear Path
- output a predicted instantaneous or mean velocity(t > r or t = t)

some trials (delete them afterwards) are also contained



In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.cuda.amp import GradScaler, autocast
import os
from PIL import Image
from tqdm.auto import tqdm
import math
import numpy as np
import random

# --- Configuration for Flow Matching ---
class FlowMatchingConfig:
    # Dataset
    data_dir = "./tiny-imagenet-200" # Path to your extracted Tiny ImageNet folder
    image_size = 64
    num_channels = 3 # RGB images

    # Training
    batch_size = 64 # Aim for this with FP16 on 4070, adjust if OOM
    num_epochs = 500 # Flow matching can converge faster than diffusion, but still needs many
    learning_rate = 1e-4
    lr_warmup_steps = 500
    mixed_precision = "fp16" # Enable mixed precision
    gradient_accumulation_steps = 1 # Increase if batch_size is too small to simulate larger batches

    # Model
    model_out_channels = 3 # The U-Net predicts a 3-channel vector field (vx, vy, vz for RGB)
    unet_block_out_channels = (64, 128, 256, 512) # Example U-Net channels
    unet_layers_per_block = 2

    # Flow Matching Specific
    # The U-Net will take a noisy image x_t and a time 't' (0 to 1)
    # The 'time_embedding_type' is handled by UNet2DModel
    # We will sample t uniformly from [0, 1]

    # Logging and Saving
    output_dir = "flow_matching_tiny_imagenet_output"
    save_image_epochs = 50
    save_model_epochs = 100
    log_interval = 100

    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = FlowMatchingConfig()

# Create output directory if it doesn't exist
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(os.path.join(config.output_dir, "samples"), exist_ok=True)
os.makedirs(os.path.join(config.output_dir, "checkpoints"), exist_ok=True)

print(f"Using device: {config.device}")

Using device: cpu


In [3]:
torch.cuda.is_available()

False

In [4]:
import torch
print(torch.__version__)
print(torch.version.cuda)



2.7.1+cu128
12.8


In [ ]:
2.7.1+cu128
12.8

In [5]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


CUDA available: False
Torch CUDA version: 12.8
Device count: 1
Device name: None


In [ ]:
CUDA available: False
Torch CUDA version: 12.8
Device count: 1
Device name: None

In [ ]:
import os
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
import torch
print("torch.cuda.device_count() =", torch.cuda.device_count())
print("torch.cuda.is_available() =", torch.cuda.is_available())








CUDA_VISIBLE_DEVICES = None
torch.cuda.device_count() = 1
torch.cuda.is_available() = True


CUDA_VISIBLE_DEVICES = None
torch.cuda.device_count() = 1
torch.cuda.is_available() = False
/home/jimmyxu/apps/miniconda3/envs/ml-project-env/lib/python3.10/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0